# Flow Matching — CIFAR-10
Classifier-Free Guidance (CFG) with a UNet velocity field trained via conditional flow matching.

| module | responsibility |
|---|---|
| `models/unet.py` | `UNet`, `ResBlock`, `SinusoidalPosEmb` |
| `data.py` | dataset & dataloader helpers |
| `trainer.py` | `EMA`, `train()` |
| `sampler.py` | `generate()`, `show_images()` |
| `evaluate.py` | `compute_fid()` |

In [ ]:
import torch
import torchvision
import matplotlib.pyplot as plt

from models.unet import UNet, NUM_CLASSES
from data import get_dataset, get_loader
from trainer import train
from sampler import generate, show_images, CIFAR10_CLASSES
from evaluate import compute_fid

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")

## Data
- CIFAR-10: 50k training images, 3×32×32
- Normalize to `[-1, 1]`

In [ ]:
dataset = get_dataset()
print(f"dataset size: {len(dataset):,}")

imgs, _ = next(iter(get_loader(dataset, batch_size=32)))
grid = torchvision.utils.make_grid(imgs * 0.5 + 0.5, nrow=8, padding=2)
plt.figure(figsize=(10, 4))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.title("Real CIFAR-10 images"); plt.axis("off"); plt.show()

In [ ]:
# Google Colab only — mount Drive for checkpoint persistence
# from google.colab import drive
# drive.mount("/content/drive")

## Model
- UNet with sinusoidal time embedding
- Class label injected alongside time (CFG: null token = index 10)
- `forward(t, x, y)` returns vector field with same shape as `x`

In [ ]:
with torch.no_grad():
    _out = UNet()(torch.rand(4), torch.randn(4, 3, 32, 32), torch.zeros(4, dtype=torch.long))
    print(f"output shape : {_out.shape}")
print(f"parameters   : {sum(p.numel() for p in UNet().parameters()) / 1e6:.1f} M")

## Training — Classifier-Free Guidance (CFG)
- Each batch passes the real class label `y` to the model
- With probability `P_UNCOND=0.15` the label is replaced by the null token (index 10)
- This trains the model to produce both conditional and unconditional vector fields
- At inference, blend them: `v = v_uncond + w * (v_cond - v_uncond)`

In [ ]:
CHECKPOINT_DIR = "checkpoints"  # change to "drive/MyDrive/..." on Colab

model, _ = train(
    epochs=100,
    lr=2e-4,
    checkpoint_dir=CHECKPOINT_DIR,
    dataset=dataset,
    device=device,
)

## Resume from Checkpoint
Checkpoints are saved to `checkpoints/ckpt_epoch<N>.pt` every 10 epochs.
Each file stores model weights, EMA weights, optimizer state, and scheduler state
so the cosine LR schedule continues seamlessly.

In [ ]:
import glob

ckpts = sorted(glob.glob(f"{CHECKPOINT_DIR}/ckpt_epoch*.pt"))
for p in ckpts:
    info = torch.load(p, map_location="cpu", weights_only=True)
    print(f"  {p}  (epoch {info['epoch']})")

RESUME_FROM = ckpts[-1] if ckpts else None
if RESUME_FROM:
    model, _ = train(
        epochs=100,
        lr=2e-4,
        checkpoint_dir=CHECKPOINT_DIR,
        resume_from=RESUME_FROM,
        dataset=dataset,
        device=device,
    )
else:
    print("No checkpoint found — run the training cell above to start fresh.")

## Inference — CFG Sampling
Each ODE step runs **two** forward passes:
- `v_cond`   = model with class label `y`
- `v_uncond` = model with null token

Guided velocity: `v = v_uncond + w × (v_cond - v_uncond)`  
Higher `w` → sharper / more class-typical images but less diversity.

In [ ]:
# 8 samples per class (80 total), laid out as 10 rows × 8 columns
y_all   = torch.arange(NUM_CLASSES, device=device).repeat_interleave(8)
samples = generate(model, y=y_all, steps=100, guidance_scale=3.0)

imgs = (samples.cpu() * 0.5 + 0.5).clamp(0, 1)
grid = torchvision.utils.make_grid(imgs, nrow=8, padding=2)
fig, ax = plt.subplots(figsize=(14, 20))
ax.imshow(grid.permute(1, 2, 0).numpy())
ax.axis("off")
h = grid.shape[1] / NUM_CLASSES
for i, name in enumerate(CIFAR10_CLASSES):
    ax.text(-4, (i + 0.5) * h, name, va="center", ha="right", fontsize=11)
plt.title("Guided Flow Matching — CIFAR-10 (guidance_scale=3)", fontsize=13)
plt.tight_layout()
plt.savefig("flow_matching_cifar10_guided.png", dpi=150)
plt.show()

In [ ]:
# Compare guidance scales: low = diverse, high = class-typical
y_dogs = torch.full((16,), CIFAR10_CLASSES.index("dog"), device=device)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for ax, w in zip(axes, [1.0, 2.0, 4.0, 7.0]):
    imgs = generate(model, y=y_dogs, steps=100, guidance_scale=w)
    imgs = (imgs.cpu() * 0.5 + 0.5).clamp(0, 1)
    grid = torchvision.utils.make_grid(imgs, nrow=4, padding=2)
    ax.imshow(grid.permute(1, 2, 0).numpy())
    ax.set_title(f"w = {w}"); ax.axis("off")
plt.suptitle("CFG guidance scale sweep (class: dog)", fontsize=13)
plt.tight_layout()
plt.savefig("flow_matching_cifar10_guidance_sweep.png", dpi=150)
plt.show()

In [ ]:
torch.save(model.state_dict(), "flow_matching_cifar10.pt")
print("model weights saved to flow_matching_cifar10.pt")

# To reload:
# model = UNet(in_ch=3, base_ch=64, ch_mult=(1, 2, 4)).to(device)
# model.load_state_dict(torch.load("flow_matching_cifar10.pt", weights_only=True))

## FID Evaluation
Generates `n_fake` random samples (class-balanced) and computes Fréchet Inception Distance
against `n_real` real CIFAR-10 images. Lower is better; a well-trained model typically reaches ~3–6.

Requires: `poetry install --with eval`

In [ ]:
fid_score = compute_fid(model, dataset)

## Comparison: Additive Injection vs AdaGN (scale+shift)

| Model | Time emb | Conditioning in ResBlock |
|---|---|---|
| **Naive** (`UNet`) | Sinusoidal → MLP | `h = h + Linear(t_emb)` |
| **AdaGN** (`UNetAdaGN`) | Sinusoidal → MLP | `h = GroupNorm(h) * (1+γ) + β`,  where γ, β = Linear(t_emb) |

Both models are trained identically. The only difference is how `t_emb` is injected into each ResBlock.

In [ ]:
from models.unet import UNetAdaGN

COMPARE_EPOCHS = 50          # increase for a more meaningful comparison
CKPT_NAIVE = "checkpoints_compare_naive"
CKPT_ADAGN = "checkpoints_compare_adagn"

p_naive = sum(p.numel() for p in UNet().parameters()) / 1e6
p_adagn = sum(p.numel() for p in UNetAdaGN().parameters()) / 1e6
print(f"UNet      parameters: {p_naive:.2f} M")
print(f"UNetAdaGN parameters: {p_adagn:.2f} M")

In [ ]:
model_naive, losses_naive = train(
    epochs=COMPARE_EPOCHS, lr=2e-4,
    checkpoint_dir=CKPT_NAIVE,
    dataset=dataset, device=device,
)

model_adagn, losses_adagn = train(
    model=UNetAdaGN(in_ch=3, base_ch=64, ch_mult=(1, 2, 4)).to(device),
    epochs=COMPARE_EPOCHS, lr=2e-4,
    checkpoint_dir=CKPT_ADAGN,
    dataset=dataset, device=device,
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(losses_naive, label="Naive (additive)")
ax.plot(losses_adagn, label="AdaGN (scale+shift)")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title(f"Training loss — {COMPARE_EPOCHS} epochs")
ax.legend()
plt.tight_layout()
plt.savefig("compare_loss_curves.png", dpi=150)
plt.show()

In [ ]:
# 4 samples per class, one row per model
y_show = torch.arange(NUM_CLASSES, device=device).repeat_interleave(4)

imgs_naive = generate(model_naive, y=y_show, steps=100, guidance_scale=3.0)
imgs_adagn = generate(model_adagn, y=y_show, steps=100, guidance_scale=3.0)

def to_grid(imgs):
    imgs = (imgs.cpu() * 0.5 + 0.5).clamp(0, 1)
    return torchvision.utils.make_grid(imgs, nrow=NUM_CLASSES, padding=2).permute(1, 2, 0)

fig, axes = plt.subplots(2, 1, figsize=(16, 4))
axes[0].imshow(to_grid(imgs_naive)); axes[0].set_title("Naive (additive)");    axes[0].axis("off")
axes[1].imshow(to_grid(imgs_adagn)); axes[1].set_title("AdaGN (scale+shift)"); axes[1].axis("off")
plt.tight_layout()
plt.savefig("compare_samples.png", dpi=150)
plt.show()

In [ ]:
# FID comparison — slow (~5 min on GPU); requires `poetry install --with eval`
fid_naive = compute_fid(model_naive, dataset, n_fake=5000, guidance_scale=3.0, steps=100)
fid_adagn = compute_fid(model_adagn, dataset, n_fake=5000, guidance_scale=3.0, steps=100)
print(f"\nNaive  FID: {fid_naive:.2f}")
print(f"AdaGN  FID: {fid_adagn:.2f}")